# Januário Leads — Fase 8 V2: Geoapify Free (piloto comercial)

**Objetivo:** buscar contatos comerciais faltantes de empresas Overture em um lote pequeno e seguro, **sem atualizar dados originais, o buscador ou a produção**.

### O que mudou após o primeiro teste (25 chamadas, zero correspondências)
- A seleção agora usa a RPC **next_overture_geoapify_business_targets**, implantada somente na homologação, e exclui rios, lagos, templos, aeroportos e órgãos públicos do piloto comercial.
- A amostra é diversificada entre categorias reais e prioriza endereços mais detalhados.
- Tenta primeiro nome e posição; se não encontrar correspondência, opcionalmente tenta uma busca sem nome, por categoria e coordenadas.
- Os motivos são separados: `zero_results`, `missing_identity`, `too_far`, `name_rejected`. Nome semelhante **e** proximidade ainda são obrigatórios para sugerir correspondência.
- Consulta **Place Details** somente quando encontrou uma correspondência e um contato está ausente. Todos os pedidos custam um crédito reservado **antes** da chamada. As chamadas de fallback e Details contam no máximo **25 por execução**.
- `RETRY_NO_MATCH = False` preserva as tentativas anteriores; ativar `True` reavalia apenas resultados `no_match` não revisados, sem apagar o histórico Overture.

### Proteções de custo e armazenamento
- **Somente HOMOLOGAÇÃO**: `prlrxjothrslhswpkoza`. Não atinge produção.
- Confirmar que a conta Geoapify está no **Free sem cartão**; o script **não tem controle sobre o faturamento da sua conta**.
- Places retorna no máximo **20 locais por chamada (1 crédito)**; Details solicita apenas o recurso básico (1 crédito). Teto compartilhado de **250 reservas diárias em UTC** para este pipeline, mais teto **25 requisições por execução**, incluindo Places, fallback e Details.
- Pausa acima de **210 MiB**. Banco Overture não é modificado: sugestões separadas em `overture_geoapify_enrichment` (`reviewed=false`) e com fonte indicada.
- Nenhuma consulta Google Places nem acesso automático a sites/Instagram. Uma sugestão não comprova titularidade da empresa.

### Executar
1. Execute as células **1 e 2** (sem consulta externa).
2. Na célula **3**, confirme `GEOAPIFY_FREE_25` e insira as duas chaves nos campos ocultos, **nunca aqui no chat**.
3. Envie somente o **RELATÓRIO FASE 8 V2** para avaliarmos a taxa real de novos contatos e a necessidade de ajustes antes de continuar.


In [ ]:
# CÉLULA 1 — configurações e utilitários puros (NÃO acessa a internet)
import datetime as dt
import difflib
import getpass
import json
import math
import re
import time
import unicodedata
from urllib.parse import urlencode, urlparse
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

STAGING = "https://prlrxjothrslhswpkoza.supabase.co"
PROJECT_REF = "prlrxjothrslhswpkoza"
GEOAPIFY_ENDPOINT = "https://api.geoapify.com/v2/places"
MODE = "no_contact"        # Depois: "any_missing" ou "all"
RETRY_NO_MATCH = False   # Só reavalia no_match quando True; preserva o histórico
MAX_REQUESTS_RUN = 25      # Máximo por execução, não aumente sem necessidade
MAX_REQUESTS_DAY = 250     # Limite GLOBAL no Supabase para este pipeline
STOP_DB_MIB = 210.0
RADIUS_METERS = 450
MATCH_MAX_DISTANCE_METERS = 220
MATCH_NAME_THRESHOLD = 0.82
CATEGORIES = "commercial,catering,healthcare,accommodation,office,service,leisure,sport,education,entertainment,tourism"
GEOAPIFY_LIMIT = 20
PAUSE_SECONDS = 1.1
assert MODE in ("no_contact","any_missing","all")
assert 1 <= MAX_REQUESTS_RUN <= 25
assert 1 <= MAX_REQUESTS_DAY <= 250
assert GEOAPIFY_LIMIT == 20 and RADIUS_METERS == 450
assert RETRY_NO_MATCH in (True, False)

def normalized(value):
    value = unicodedata.normalize("NFKD",str(value or ""))
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9]+"," ",value.lower())).strip()

def business_tokens(name):
    stop={"de","da","do","das","dos","e","ltda","eireli","me","sa"}
    return [v for v in normalized(name).split() if v not in stop]

def name_similarity(a,b):
    plain=sim(a,b)
    left,right=business_tokens(a),business_tokens(b)
    if len(left)>=2 and len(right)>=2:
        return max(plain,difflib.SequenceMatcher(None," ".join(sorted(left))," ".join(sorted(right))).ratio())
    return plain

def sim(a,b):
    x,y=normalized(a),normalized(b)
    if not x or not y: return 0.0
    return difflib.SequenceMatcher(None,x,y).ratio()

def haversine(a,b,c,d):
    x1,y1,x2,y2=map(math.radians,(a,b,c,d))
    delta_x,delta_y=x2-x1,y2-y1
    q=math.sin(delta_x/2)**2+math.cos(x1)*math.cos(x2)*math.sin(delta_y/2)**2
    return 2*6371000*math.asin(min(1,math.sqrt(q)))

def clean_phone(value):
    if not isinstance(value,str): return None
    raw=re.sub(r"[^0-9+]","",value.strip())[:28]
    digits=re.sub(r"\D","",raw)
    local=digits[2:] if digits.startswith("55") and len(digits)>=12 else digits
    return raw if re.fullmatch(r"[0-9]{10,11}",local) else None

def clean_email(value):
    if not isinstance(value,str):return None
    e=value.strip().split(";")[0].split(",")[0].lower()[:180]
    return e if re.fullmatch(r"[^@\s]+@[^@\s]+\.[a-z]{2,}",e) else None

def clean_web(value):
    if not isinstance(value,str) or len(value)>400:return None
    v=value.strip()
    if v.startswith("www."):v="https://"+v
    try:
        u=urlparse(v)
        host=(u.hostname or "").lower()
        if u.scheme not in ("https","http") or not host or "." not in host or u.username or u.password:
            return None
        if host in ("localhost","example.com") or host.endswith((".local",".internal",".test")):
            return None
        return v
    except ValueError:
        return None

def clean_instagram(value):
    if not isinstance(value,str):return None
    s=value.strip()
    if s.startswith("@"):s="https://www.instagram.com/"+s[1:]
    elif s.lower().startswith("instagram.com/"):s="https://www."+s
    elif s.lower().startswith("www.instagram.com/"):s="https://"+s
    try:
        u=urlparse(s)
        if u.scheme not in ("http","https") or (u.hostname or "").lower() not in ("instagram.com","www.instagram.com"):
            return None
        parts=[x for x in u.path.split("/") if x]
        if len(parts)!=1 or parts[0].lower() in ("p","reel","stories","explore","accounts"):
            return None
        if not re.fullmatch(r"[a-zA-Z0-9_.]{2,30}",parts[0]):
            return None
        return "https://www.instagram.com/"+parts[0]+"/"
    except ValueError:
        return None

def _first(values,fn):
    for value in values:
        out=fn(value)
        if out:return out
    return None

def candidate_fields(feature):
    p=feature.get("properties",{}) if isinstance(feature,dict) else {}
    if not isinstance(p,dict):return {}
    contact=p.get("contact") if isinstance(p.get("contact"),dict) else {}
    ds=p.get("datasource") if isinstance(p.get("datasource"),dict) else {}
    raw=ds.get("raw") if isinstance(ds.get("raw"),dict) else {}
    coords=(feature.get("geometry") or {}).get("coordinates",[])
    if not isinstance(coords,list) or len(coords)<2 or not all(isinstance(i,(int,float)) for i in coords[:2]):return {}
    email=_first((contact.get("email"),p.get("email"),raw.get("contact:email"),raw.get("email")),clean_email)
    website=_first((p.get("website"),contact.get("website"),raw.get("contact:website"),raw.get("website")),clean_web)
    phone=_first((contact.get("phone"),p.get("phone"),raw.get("contact:phone"),raw.get("phone")),clean_phone)
    instagram=_first((contact.get("instagram"),p.get("instagram"),raw.get("contact:instagram"),raw.get("instagram")),clean_instagram)
    address=p.get("formatted") or p.get("address_line1")
    category=p.get("categories")
    return {"name":p.get("name") or raw.get("name"),"lon":float(coords[0]),"lat":float(coords[1]),
            "place_id":p.get("place_id"),"phone":phone,"email":email,"website":website,
            "instagram":instagram,"address":str(address)[:400] if isinstance(address,str) else None,
            "categories":[str(x)[:100] for x in category[:10]] if isinstance(category,list) else []}

def best_match(overture,features):
    matches=[]
    for f in features:
        x=candidate_fields(f)
        if not x.get("name") or not x.get("place_id"):continue
        d=haversine(float(overture["latitude"]),float(overture["longitude"]),x["lat"],x["lon"])
        name_score=name_similarity(overture["name"],x["name"])
        if d>MATCH_MAX_DISTANCE_METERS or name_score<MATCH_NAME_THRESHOLD:
            continue
        x["distance_m"]=int(round(d))
        x["name_similarity"]=round(name_score,4)
        matches.append(x)
    return max(matches,key=lambda x:(x["name_similarity"],-x["distance_m"]),default=None)

def rest(secret,path,method="GET",payload=None,prefer=None):
    assert path.startswith("/rest/v1/") and not path.startswith("//")
    headers={"apikey":secret,"Accept":"application/json","Content-Type":"application/json"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    if prefer:headers["Prefer"]=prefer
    raw=json.dumps(payload,ensure_ascii=False).encode() if payload is not None else None
    req=Request(STAGING+path,data=raw,headers=headers,method=method)
    with urlopen(req,timeout=25) as response:
        body=response.read(1_000_000)
        return json.loads(body.decode("utf-8")) if body else None

CATEGORY_HINTS={
    "barber":"service","beauty_salon":"service",
    "restaurant":"catering","pizza_restaurant":"catering","bar":"catering",
    "bakery":"commercial,catering","cafe":"catering","coffee_shop":"catering",
    "hotel":"accommodation","gym":"sport,leisure",
    "automotive_repair":"service","car_wash":"service",
    "pharmacy":"commercial,healthcare","dental_clinic":"healthcare",
    "medical_clinic":"healthcare",
    "professional_service":"office,service","attorney_or_law_firm":"office",
}
GEOAPIFY_DETAILS_ENDPOINT="https://api.geoapify.com/v2/place-details"

def geoapify_places(key,target,with_name=True):
    # Uma chamada por estratégia; o fallback sem nome exige outro crédito reservado.
    lon,lat=float(target["longitude"]),float(target["latitude"])
    params={"apiKey":key,"categories":CATEGORY_HINTS.get(target.get("category"),CATEGORIES),
            "filter":"circle:{:.6f},{:.6f},{}".format(lon,lat,RADIUS_METERS),
            "bias":"proximity:{:.6f},{:.6f}".format(lon,lat),
            "limit":str(GEOAPIFY_LIMIT),"lang":"pt"}
    if with_name:params["name"]=target["name"][:150]
    request=Request(GEOAPIFY_ENDPOINT+"?"+urlencode(params),headers={"Accept":"application/json"})
    with urlopen(request,timeout=15) as response:
        payload=response.read(1_000_000)
        document=json.loads(payload.decode())
        if not isinstance(document,dict) or not isinstance(document.get("features"),list):
            raise ValueError("resposta Geoapify Places inesperada")
        return document["features"]

def geoapify_details(key,place_id):
    # Apenas details (1 crédito); não solicita building, geometrias extras ou isoáreas.
    query=urlencode({"apiKey":key,"id":str(place_id),"features":"details","lang":"pt"})
    request=Request(GEOAPIFY_DETAILS_ENDPOINT+"?"+query,headers={"Accept":"application/json"})
    with urlopen(request,timeout=15) as response:
        document=json.loads(response.read(1_000_000).decode())
        if not isinstance(document,dict) or not isinstance(document.get("features"),list):
            raise ValueError("resposta Geoapify Place Details inesperada")
        return document["features"]

def details_fields(features):
    # Place Details pode retornar polígonos; contatos não dependem da geometria.
    out={}
    for feature in features:
        p=feature.get("properties",{}) if isinstance(feature,dict) else {}
        if not isinstance(p,dict) or p.get("feature_type") not in (None,"details"):continue
        contact=p.get("contact") if isinstance(p.get("contact"),dict) else {}
        datasource=p.get("datasource") if isinstance(p.get("datasource"),dict) else {}
        raw=datasource.get("raw") if isinstance(datasource.get("raw"),dict) else {}
        options={
          "phone":_first((contact.get("phone"),p.get("phone"),raw.get("contact:phone"),raw.get("phone")),clean_phone),
          "email":_first((contact.get("email"),p.get("email"),raw.get("contact:email"),raw.get("email")),clean_email),
          "website":_first((p.get("website"),contact.get("website"),raw.get("contact:website"),raw.get("website")),clean_web),
          "instagram":_first((contact.get("instagram"),p.get("instagram"),raw.get("contact:instagram"),raw.get("instagram")),clean_instagram),
        }
        for field,val in options.items():
            if val and not out.get(field):out[field]=val
    return out

def match_diagnostic(overture,features):
    if not features:return None,"zero_results"
    viable=[candidate_fields(f) for f in features]
    viable=[v for v in viable if v.get("name") and v.get("place_id")]
    if not viable:return None,"missing_identity"
    near=[v for v in viable if haversine(float(overture["latitude"]),float(overture["longitude"]),v["lat"],v["lon"])<=MATCH_MAX_DISTANCE_METERS]
    if not near:return None,"too_far"
    matched=best_match(overture,features)
    return (matched,"matched") if matched else (None,"name_rejected")

def quota_credit(secret):
    out=rest(secret,"/rest/v1/rpc/claim_overture_geoapify_free_credit",
             "POST",{"p_daily_limit":MAX_REQUESTS_DAY})
    return out is True

def storage_mib(secret):
    value=rest(secret,"/rest/v1/rpc/overture_geoapify_storage_mib","POST",{})
    return float(value)

def next_targets(secret,count):
    return rest(secret,"/rest/v1/rpc/next_overture_geoapify_business_targets",
                "POST",{"p_mode":MODE,"p_limit":count,"p_retry_no_match":RETRY_NO_MATCH})

def save_suggestion(secret,row):
    rest(secret,"/rest/v1/overture_geoapify_enrichment?on_conflict=overture_id",
         "POST",row,prefer="resolution=merge-duplicates,return=minimal")

def status_counts(secret):
    for path in (
        "/rest/v1/overture_geoapify_enrichment?select=status",
        "/rest/v1/overture_geoapify_free_credit_usage?select=utc_day,credits_reserved&order=utc_day.desc&limit=2"
    ):
        # Avoid scanning the table of all companies just to build the summary.
        if "select=status" in path: continue
        return rest(secret,path)
print("Funções prontas. Nenhuma chave ou consulta externa utilizada.")


In [ ]:
# CÉLULA 2 — testes locais, sem consumir créditos
assert normalized("São João Barber")=="sao joao barber"
assert sim("Barbearia do Carlos","Barbearia do Carlos") == 1.0
assert sim("Barbearia do Carlos","Restaurante Pedro") < MATCH_NAME_THRESHOLD
assert round(haversine(-21.25,-45.0,-21.25,-45.0))==0
assert clean_email("contato@exemplo.com")=="contato@exemplo.com"
assert clean_instagram("https://www.instagram.com/barbearia_carlos/")=="https://www.instagram.com/barbearia_carlos/"
assert clean_instagram("https://instagram.com/p/ABCDE")==None
assert candidate_fields({"properties":{},"geometry":{"coordinates":[-45.,-21.]}}).get("place_id") is None
sample={"id":"teste","name":"Barbearia do Carlos","latitude":-21.25,"longitude":-45.0,
        "phone":None,"email":None,"website":None,"instagram":None}
feature={"properties":{"name":"Barbearia do Carlos","place_id":"geo-test",
        "contact":{"email":"contato@exemplo.com","phone":"+5535999999999",
        "website":"https://www.barber.com.br","instagram":"@barber_carlos"}},
        "geometry":{"coordinates":[-45.0,-21.25]}}
assert best_match(sample,[feature])["place_id"]=="geo-test"
wrong={"properties":{"name":"Padaria Geral","place_id":"geo-wrong"},
       "geometry":{"coordinates":[-45.0,-21.25]}}
assert best_match(sample,[wrong]) is None
far={"properties":{"name":"Barbearia do Carlos","place_id":"geo-far"},
       "geometry":{"coordinates":[-45.05,-21.25]}}
assert best_match(sample,[far]) is None
assert name_similarity("João Barber", "Barber João") >= MATCH_NAME_THRESHOLD
assert name_similarity("Mercearia Do Zé","Barbearia Do Zé") < MATCH_NAME_THRESHOLD
assert match_diagnostic(sample,[]) == (None,"zero_results")
assert match_diagnostic(sample,[wrong])[1]=="name_rejected"
assert match_diagnostic(sample,[far])[1]=="too_far"
assert match_diagnostic(sample,[feature])[1]=="matched"
details={"features":[{"properties":{"feature_type":"details","contact":{"phone":"+5535999999999","email":"contato@exemplo.com"},"website":"https://www.barber.com.br"},"geometry":{"type":"Polygon","coordinates":[]}}]}
assert details_fields(details["features"])["email"]=="contato@exemplo.com"
assert details_fields(details["features"])["phone"]=="+5535999999999"
assert CATEGORY_HINTS["restaurant"]=="catering"
print("TESTES APROVADOS: validação de nome, posição, e-mail, Instagram e proteção contra falsos positivos.")


In [ ]:
# CÉLULA 3 — execução MANUAL, com limite de 25 requisições externas somadas
def run_geoapify_free():
    print("FASE 8 V2 — pesquisa comercial controlada, sem Google Places.")
    print("Modo:",MODE,"| reavaliar no_match:",RETRY_NO_MATCH,
          "| teto de CHAMADAS por execução:",MAX_REQUESTS_RUN,
          "| teto diário do pipeline:",MAX_REQUESTS_DAY)
    print("A conta Geoapify precisa estar no plano Free sem cartão e com orçamento seguro.")
    confirm=input("Digite GEOAPIFY_FREE_25 para confirmar: ").strip()
    if confirm!="GEOAPIFY_FREE_25":
        print("CANCELADO. Nenhuma consulta externa feita.")
        return
    key=getpass.getpass("Chave SECRETA do Supabase HOMOLOGAÇÃO (campo oculto): ").strip()
    api=getpass.getpass("Chave do Geoapify FREE (campo oculto): ").strip()
    if not (key.startswith("sb_secret_") or key.startswith("eyJ")) or not api or len(api)>256:
        print("Credencial inválida ou ausente. Execução encerrada.")
        return
    calls=0
    abort_reason=None
    processed=0
    matched=0
    no_match=0
    fallback_calls=0
    details_calls=0
    extra_fields={"phone":0,"email":0,"website":0,"instagram":0,"address":0}
    diagnostics={"zero_results":0,"missing_identity":0,"too_far":0,"name_rejected":0}

    try:
        size=storage_mib(key)
        print("Banco de HOMOLOGAÇÃO:",round(size,2),"MiB")
        if size>=STOP_DB_MIB:
            print("BLOQUEADO: limite preventivo do banco atingido.")
            return
        targets=next_targets(key,MAX_REQUESTS_RUN)
        if not isinstance(targets,list):
            print("Alvos indisponíveis no Supabase.")
            return
        print("Empresas comerciais elegíveis neste lote:",len(targets))

        def dispatch(fn,*args):
            # Reservar 1 crédito atômico antes de CADA chamada: Places, fallback ou Details.
            nonlocal calls,abort_reason
            if calls>=MAX_REQUESTS_RUN:
                return None  # sem chamar o provedor; permite salvar resultado parcial
            if storage_mib(key)>=STOP_DB_MIB:
                abort_reason="limite de armazenamento preventivo"
                return None
            if not quota_credit(key):
                abort_reason="limite diário de 250 créditos atingido"
                return None
            calls+=1
            try:
                return fn(api,*args)
            except HTTPError as error:
                abort_reason="HTTP "+str(error.code)+" do Geoapify"
            except (URLError,TimeoutError,ValueError) as error:
                abort_reason=type(error).__name__+" na consulta Geoapify"
            return None

        for item in targets:
            if calls>=MAX_REQUESTS_RUN or abort_reason:break
            if not isinstance(item,dict) or not item.get("id") or not item.get("name"):
                continue
            # Etapa 1: pesquisa pelo nome. O raio da pesquisa não equivale à
            # distância máxima do matching, que permanece conservadora.
            features=dispatch(geoapify_places,item,True)
            if features is None:break
            candidate,named_reason=match_diagnostic(item,features)
            fallback_reason="not_needed"
            if candidate is None and not abort_reason and calls<MAX_REQUESTS_RUN:
                # Etapa 2: pesquisa por categoria/posição sem nome.
                # Usa um crédito adicional somente se necessário.
                nearby=dispatch(geoapify_places,item,False)
                if nearby is not None:
                    fallback_calls+=1
                    candidate,fallback_reason=match_diagnostic(item,nearby)
            if abort_reason:break
            reason="matched" if candidate else (fallback_reason if fallback_reason!="not_needed" else named_reason)

            record={"overture_id":item["id"],
                    "status":"matched" if candidate else "no_match",
                    "checked_at":dt.datetime.now(dt.timezone.utc).isoformat(),
                    "reviewed":False}
            if candidate:
                def missing(field):return not str(item.get(field) or "").strip()
                detail_status="not_needed"
                if calls<MAX_REQUESTS_RUN and any(missing(f) and not candidate.get(f) for f in ("phone","email","website","instagram")):
                    d=dispatch(geoapify_details,candidate["place_id"])
                    if d is not None:
                        details_calls+=1
                        for field,value in details_fields(d).items():
                            if value and not candidate.get(field):candidate[field]=value
                        detail_status="queried"
                    elif abort_reason:
                        # Não descartar uma correspondência já confirmada pelo Places.
                        detail_status="interrupted"
                record.update({
                  "candidate_name":str(candidate["name"])[:180],
                  "candidate_address":candidate["address"] if missing("address") else None,
                  "candidate_phone":candidate["phone"] if missing("phone") else None,
                  "candidate_email":candidate["email"] if missing("email") else None,
                  "candidate_website":candidate["website"] if missing("website") else None,
                  "candidate_instagram":candidate["instagram"] if missing("instagram") else None,
                  "geoapify_place_id":str(candidate["place_id"])[:256],
                  "geoapify_categories":candidate["categories"],
                  "matched_distance_m":candidate["distance_m"],
                  "matched_name_similarity":candidate["name_similarity"],
                  "note":"V2; busca_nome="+named_reason+"; busca_sem_nome="+fallback_reason+
                         "; details="+detail_status+
                         "; origem=Geoapify Places/Details; confirmar titularidade e contatos antes de uso",
                })
            else:
                record["note"]=("V2; busca_nome="+named_reason+"; busca_sem_nome="+fallback_reason+
                                "; motivo_final="+reason+"; Geoapify Places")
            try:
                save_suggestion(key,record)
            except HTTPError as error:
                abort_reason="falha ao salvar no banco: HTTP "+str(error.code)
                break
            processed+=1
            if candidate:
                matched+=1
                new_fields=0
                for field in extra_fields:
                    if record.get("candidate_"+field):
                        extra_fields[field]+=1
                        new_fields+=1
                print("CANDIDATO:",item.get("name","")[:55],item.get("city",""),
                      item.get("state",""),"| distância:",candidate["distance_m"],
                      "m | novos campos:",new_fields)
            else:
                no_match+=1
                if reason in diagnostics:diagnostics[reason]+=1
                print("SEM CORRESPONDÊNCIA:",item.get("name","")[:55],item.get("city",""),
                      item.get("state",""),"| motivo:",reason,
                      "| tentativa por nome:",named_reason)
            if abort_reason:break
            time.sleep(PAUSE_SECONDS)

        print("\n"+"="*58)
        print("RELATÓRIO FASE 8 V2 — Geoapify Free")
        print("Chamadas Geoapify reservadas:",calls,"(teto: 25, inclusive fallback e Details)")
        print("Empresas processadas:",processed)
        print("Correspondências candidatas:",matched)
        print("Sem correspondência:",no_match)
        print("Pesquisas adicionais sem nome:",fallback_calls)
        print("Consultas Place Details:",details_calls)
        for reason,count in diagnostics.items():
            print("Diagnóstico",reason+":",count)
        print("Novos telefones sugeridos:",extra_fields["phone"])
        print("Novos e-mails sugeridos:",extra_fields["email"])
        print("Novos sites sugeridos:",extra_fields["website"])
        print("Novos Instagrams sugeridos:",extra_fields["instagram"])
        print("Novos endereços sugeridos:",extra_fields["address"])
        print("Banco após execução:",round(storage_mib(key),2),"MiB")
        print("Google Places: ZERO requisições.")
        print("Dados Overture: intocados. Sugestões são revisáveis.")
        if abort_reason:print("PAUSA:",abort_reason)
        print("Credenciais devem permanecer privadas.")
        print("="*58)
    except HTTPError as error:
        print("Erro HTTP do Supabase HOMOLOGAÇÃO:",error.code,"— nenhuma chave exibida.")
    finally:
        key=""
        api=""
        print("Credenciais descartadas desta execução. Desconecte o runtime ao terminar.")

run_geoapify_free()


## Avalie antes de repetir
Veja a proporção de empresas processadas e os diagnósticos (zero resultados, rejeição por nome ou distância) juntamente com os novos contatos sugeridos. **Não** conclua que todas as empresas têm dados no Geoapify só porque houve correspondência. Não automatize lotes nacionais sem avaliação de custo, licenciamento e qualidade.

**Atenção:** a primeira versão fazia até 25 *empresas* por execução. A V2 limita rigorosamente a **25 chamadas externas**, portanto o número de empresas processadas pode ser menor porque fallback e Place Details também consomem créditos.
